In [10]:
import json
import os
import re
from pathlib import Path
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape

BASE_DIR = Path('/Users/narikenzhe/Desktop')
WORK_DIR = BASE_DIR / 'My_Jupyter'
CLUSTERS_FOLDER = WORK_DIR / 'кластера 2026'
GEO_FILE = Path('/Users/narikenzhe/Desktop/Work_data/adsl_yandex_geo.csv')
ADSL_FILE_CANDIDATES = [
    BASE_DIR / 'adsl_geo_202603301610.csv',
    WORK_DIR / 'adsl_geo_202603301610.csv',
    BASE_DIR / 'Work_data' / 'adsl_geo_202603301610.csv'
]
METRIC_CRS = 32639
OUTPUT_XLSX = WORK_DIR / f'df_comparison_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.xlsx'


In [11]:
def extract_js_literal(text: str, variable_name: str):
    marker = f"const {variable_name} ="
    start_idx = text.find(marker)
    if start_idx == -1: return None
    idx = start_idx + len(marker)
    while idx < len(text) and text[idx].isspace(): idx += 1
    if idx >= len(text): return None
    opening = text[idx]
    closing = '}' if opening == '{' else ']'
    depth = 0
    in_string, escaped, quote_char = False, False, ''
    for end_idx in range(idx, len(text)):
        char = text[end_idx]
        if in_string:
            if escaped: escaped = False
            elif char == '\\': escaped = True
            elif char == quote_char: in_string = False
            continue
        if char in ('\"', "'"): in_string, quote_char = True, char
        elif char == opening: depth += 1
        elif char == closing:
            depth -= 1
            if depth == 0:
                try: return json.loads(text[idx:end_idx + 1])
                except: return None
    return None

def coord_key(lat, lon):
    try: return (round(float(lat), 6), round(float(lon), 6))
    except: return None

def extract_cluster_data_from_folder(folder_path):
    polygons = []
    points = []
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"Папка не найдена: {folder}")

    for name in sorted(os.listdir(folder)):
        if not name.endswith('.html'): continue
        file_path = folder / name
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
            
            # Polygons
            c_data = extract_js_literal(text, 'clustersData')
            if c_data:
                for f_geo in c_data.get('features', []):
                    props = f_geo.get('properties', {}).copy()
                    if f_geo.get('geometry'):
                        props['source_file'] = name
                        props['geometry'] = shape(f_geo['geometry'])
                        polygons.append(props)
            
            # Points
            p_data = extract_js_literal(text, 'pointsData')
            if p_data:
                for f_geo in p_data.get('features', []):
                    props = f_geo.get('properties', {}).copy()
                    coords = f_geo.get('geometry', {}).get('coordinates', [])
                    if len(coords) == 2:
                        props['source_file'] = name
                        props['point_lon'], props['point_lat'] = coords[0], coords[1]
                        props['coord_key'] = coord_key(coords[1], coords[0])
                        points.append(props)

    gdf_poly = gpd.GeoDataFrame(polygons, geometry='geometry', crs='EPSG:4326') if polygons else gpd.GeoDataFrame()
    df_pts = pd.DataFrame(points) if points else pd.DataFrame()
    if not df_pts.empty: df_pts = df_pts.drop_duplicates(subset=['coord_key'])
    return gdf_poly, df_pts

def resolve_existing_path(*candidates, label='file'):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists(): return candidate
    checked = '\n'.join(f'- {c}' for c in candidates)
    raise FileNotFoundError(f'Не найден {label}. Проверены пути:\n{checked}')

def to_metric_geodataframe(df, lon_col, lat_col, epsg=METRIC_CRS):
    return gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs='EPSG:4326'
    ).to_crs(epsg=epsg)


In [12]:
ADSL_FILE = resolve_existing_path(*ADSL_FILE_CANDIDATES, label='ADSL file')

print("Загрузка CSV файлов...")
df_geo = pd.read_csv(GEO_FILE, sep=';', encoding='cp1251')
df_adsl_geo = pd.read_csv(ADSL_FILE, sep=';', encoding='cp1251')

if 'address' not in df_geo.columns:
    if 'upper_abonent_adr' in df_geo.columns:
        df_geo['address'] = df_geo['upper_abonent_adr'].fillna('')

print("Загрузка данных кластеров из папки...")
gdf_clusters, df_cluster_points = extract_cluster_data_from_folder(CLUSTERS_FOLDER)

print(f'Загружено GEO-записей: {len(df_geo)}')
print(f'Загружено ADSL-записей: {len(df_adsl_geo)}')
print(f'Собрано полигонов кластеров: {len(gdf_clusters)}')
print(f'Собрано уникальных точек из кластеров: {len(df_cluster_points)}')

# Подготовка coord_key
df_geo['coord_key'] = df_geo.apply(lambda r: coord_key(r['point_lat'], r['point_lon']), axis=1)
df_adsl_geo['coord_key'] = df_adsl_geo.apply(lambda r: coord_key(r['point_lat'], r['point_lon']), axis=1)


Загрузка CSV файлов...
Загрузка данных кластеров из папки...
Загружено GEO-записей: 1811342
Загружено ADSL-записей: 215969
Собрано полигонов кластеров: 248
Собрано уникальных точек из кластеров: 218827


In [13]:
# 1. Обработка df_geo (сопоставление с кластерами)
logging.info("--- Обработка df_geo ---")

# Инициализация колонок
df_geo_last = df_geo.copy()
df_geo_last['name'] = None
df_geo_last['source_file'] = None
df_geo_last['location_flag'] = 'вне'
df_geo_last['distance_meters'] = 9999.0

# А. Точное совпадение по coord_key с точками из кластеров
if not df_cluster_points.empty:
    logging.info("Точное сопоставление df_geo с точками кластеров...")
    # Мержим только нужные колонки
    match_exact = df_cluster_points[['coord_key', 'cluster', 'source_file']].rename(columns={'cluster': 'name_exact', 'source_file': 'file_exact'})
    df_geo_last = df_geo_last.merge(match_exact, on='coord_key', how='left')
    
    mask_exact = df_geo_last['name_exact'].notna()
    df_geo_last.loc[mask_exact, 'name'] = df_geo_last.loc[mask_exact, 'name_exact']
    df_geo_last.loc[mask_exact, 'source_file'] = df_geo_last.loc[mask_exact, 'file_exact']
    df_geo_last.loc[mask_exact, 'location_flag'] = 'внутри'
    df_geo_last.loc[mask_exact, 'distance_meters'] = 0.0
    df_geo_last = df_geo_last.drop(columns=['name_exact', 'file_exact'])

# Б. Пространственный поиск для тех, кто еще не найден
mask_unmatched = df_geo_last['name'].isna()
if mask_unmatched.any() and not gdf_clusters.empty:
    logging.info("Пространственный поиск для оставшихся точек...")
    unique_unmatched = df_geo_last[mask_unmatched][['point_lat', 'point_lon']].drop_duplicates().dropna()
    if not unique_unmatched.empty:
        gdf_unmatched = to_metric_geodataframe(unique_unmatched, 'point_lon', 'point_lat')
        gdf_clusters_metric = gdf_clusters.to_crs(epsg=METRIC_CRS)
        
        # Внутри
        inside = gpd.sjoin(gdf_unmatched, gdf_clusters_metric[['name', 'source_file', 'geometry']], how='inner', predicate='intersects')
        inside['dist_poly'] = 0.0
        inside['flag_poly'] = 'внутри'
        
        # Рядом
        outside = gdf_unmatched.loc[~gdf_unmatched.index.isin(inside.index)]
        nearby = gpd.sjoin_nearest(outside, gdf_clusters_metric[['name', 'source_file', 'geometry']], max_distance=100, how='inner', distance_col='dist_poly')
        nearby['flag_poly'] = 'рядом'
        
        geo_poly = pd.concat([inside, nearby])
        geo_poly = geo_poly.drop_duplicates(subset=['point_lat', 'point_lon'])
        
        df_geo_last = df_geo_last.merge(
            geo_poly[['point_lat', 'point_lon', 'name', 'source_file', 'dist_poly', 'flag_poly']],
            on=['point_lat', 'point_lon'],
            how='left',
            suffixes=('', '_sjoin')
        )
        
        mask_sjoin = df_geo_last['name_sjoin'].notna() & df_geo_last['name'].isna()
        df_geo_last.loc[mask_sjoin, 'name'] = df_geo_last.loc[mask_sjoin, 'name_sjoin']
        df_geo_last.loc[mask_sjoin, 'source_file'] = df_geo_last.loc[mask_sjoin, 'source_file_sjoin']
        df_geo_last.loc[mask_sjoin, 'location_flag'] = df_geo_last.loc[mask_sjoin, 'flag_poly']
        df_geo_last.loc[mask_sjoin, 'distance_meters'] = df_geo_last.loc[mask_sjoin, 'dist_poly']
        
        df_geo_last = df_geo_last.drop(columns=['name_sjoin', 'source_file_sjoin', 'dist_poly', 'flag_poly'])

logging.info(f"Итого df_geo внутри/рядом: {len(df_geo_last[df_geo_last['location_flag'].isin(['внутри', 'рядом'])])}")
display(df_geo_last.head())

2026-04-28 13:34:54,209 - INFO - --- Обработка df_geo ---
2026-04-28 13:34:54,682 - INFO - Точное сопоставление df_geo с точками кластеров...
2026-04-28 13:34:55,954 - INFO - Пространственный поиск для оставшихся точек...
2026-04-28 13:34:57,753 - INFO - Итого df_geo внутри/рядом: 244273


,customer_account_id,customer_name,town_state_name,street_name,house,address,point_lat,point_lon,coord_source,coord_key,name,source_file,location_flag,distance_meters
0,5,"РЕБОО ""ЦЕНТР ЗАБОТА - ХЭСЭД ПОЛИНА""",г.Алматы,Мукагали Макатаева,88,"улица Мукагали Макатаева, 88",43.263573,76.942788,yandex,"(43.263573, 76.942788)",None,None,вне,9999.000000
1,12,"ТОО ""АГЕНТСТВО ИНТЕЛЛЕКТУАЛЬНОЙ СОБСТВЕННОСТИ ...",г.Алматы,Казыбек би,65,"улица Казыбек би, 65",43.256916,76.945896,yandex,"(43.256916, 76.945896)",None,None,вне,9999.000000
2,23,"ТОО ""AK CONSTRUCTION COMPANY""",г.Алматы,ТАУГУЛЬ-2,13,"микро/рай ТАУГУЛЬ-2, 13 СЧЕТ",43.207047,76.883379,sw,"(43.207047, 76.883379)",Кластер 20,almaty_cluster_map.html,внутри,0.000000
3,23,"ТОО ""AK CONSTRUCTION COMPANY""",г.Алматы,ТАУГУЛЬ-2,13,"микро/рай ТАУГУЛЬ-2, 13",43.207502,76.883616,yandex,"(43.207502, 76.883616)",Кластер 20,almaty_cluster_map.html,рядом,14.250002
4,23,"ТОО ""AK CONSTRUCTION COMPANY""",г.Алматы,Желмая,31,"улица Желмая, 31",43.281178,76.981074,yandex,"(43.281178, 76.981074)",Кластер 11,almaty_cluster_map.html,внутри,0.000000


In [14]:
# 2. Обработка df_adsl_geo (синхронизация с df_geo)
logging.info("--- Обработка df_adsl_geo ---")

# Синхронизация по coord_key
cols_to_map = ['coord_key', 'name', 'source_file', 'distance_meters', 'location_flag', 'address']
geo_map = df_geo_last[cols_to_map].drop_duplicates(subset=['coord_key'])

df_adsl_last = df_adsl_geo.merge(geo_map, on='coord_key', how='left')

df_adsl_last['location_flag'] = df_adsl_last['location_flag'].fillna('вне')
logging.info(f"Итого df_adsl_geo внутри/рядом: {len(df_adsl_last[df_adsl_last['location_flag'].isin(['внутри', 'рядом'])])}")
display(df_adsl_last.head())

2026-04-28 13:34:57,762 - INFO - --- Обработка df_adsl_geo ---
2026-04-28 13:34:58,334 - INFO - Итого df_adsl_geo внутри/рядом: 54536


,customer_account_id,point_lat,point_lon,usluga_tech,gts_sts,rn,coord_key,name,source_file,distance_meters,location_flag,address
0,31776,43.213370,76.966369,adsl,ГТС,1,"(43.21337, 76.966369)",None,None,9999.0,вне,"проспект Достык, 302А"
1,59744,43.321029,76.910997,adsl,ГТС,1,"(43.321029, 76.910997)",None,None,9999.0,вне,"микро/рай АЙНАБУЛАК 1, 22В"
2,59797,43.204262,76.849435,adsl,ГТС,1,"(43.204262, 76.849435)",None,None,9999.0,вне,"микро/рай Таугуль, ул. Ыкылас, 8А"
3,59844,43.202751,76.910063,adsl,ГТС,1,"(43.202751, 76.910063)",None,None,9999.0,вне,"улица Таира Жарокова, 350"
4,60095,43.173678,76.868462,adsl,ГТС,1,"(43.173678, 76.868462)",Кластер 6,almaty_cluster_map.html,0.0,внутри,"микро/рай Каргалы, ул. Кенесары хана, 70"


In [15]:
# 3. Сравнение абонентов
logging.info("--- Сравнение абонентов из кластеров (df_geo vs df_adsl_geo) ---")

# Оставляем только уникальных абонентов из ADSL
df_adsl_unique = df_adsl_last.drop_duplicates(subset=['customer_account_id'])
logging.info(f"Уникальных абонентов в ADSL: {len(df_adsl_unique)}")

# Сравниваем по coord_key (надежнее чем по lat/lon)
df_comparison = df_geo_last.merge(
    df_adsl_unique,
    on='coord_key',
    how='outer',
    indicator='состояние_абонента',
    suffixes=('_geo', '_adsl')
)

indicator_map = {
    'left_only': 'В кластере (df_geo), но нет в ADSL',
    'right_only': 'Есть в ADSL (df_adsl_geo), но нет в df_geo',
    'both': 'ADSL абонент найден в базе GEO'
}
df_comparison['состояние_абонента'] = df_comparison['состояние_абонента'].map(indicator_map)

# Фильтруем только тех, кто в кластерах (внутри или рядом)
df_comparison = df_comparison[
    (df_comparison['location_flag_geo'].isin(['внутри', 'рядом'])) | 
    (df_comparison['location_flag_adsl'].isin(['внутри', 'рядом']))
].copy()

df_comparison['cluster_name'] = df_comparison['name_geo'].fillna(df_comparison['name_adsl'])
df_comparison['address'] = df_comparison['address_geo'].fillna(df_comparison['address_adsl'])

# ID абонента
df_comparison['customer_account_id'] = df_comparison['customer_account_id_adsl'].fillna(df_comparison['customer_account_id_geo'])
df_comparison['customer_account_id'] = pd.to_numeric(df_comparison['customer_account_id'], errors='coerce').astype('Int64')

# Удаление дубликатов по ID
len_before = len(df_comparison)
df_comparison = df_comparison.drop_duplicates(subset=['customer_account_id'])
logging.info(f"Удалено дубликатов по ID: {len_before - len(df_comparison)}. Итого: {len(df_comparison)}")

display(df_comparison['состояние_абонента'].value_counts())


2026-04-28 13:34:58,341 - INFO - --- Сравнение абонентов из кластеров (df_geo vs df_adsl_geo) ---
2026-04-28 13:34:58,381 - INFO - Уникальных абонентов в ADSL: 215969
2026-04-28 13:35:02,486 - INFO - Удалено дубликатов по ID: 164234. Итого: 202814


состояние_абонента
В кластере (df_geo), но нет в ADSL            148570
ADSL абонент найден в базе GEO                 54244
Есть в ADSL (df_adsl_geo), но нет в df_geo         0
Name: count, dtype: int64

In [16]:
# 4. Выборка колонок и сохранение
logging.info("Подготовка и сохранение результатов...")

cols = [
    'customer_account_id', 'состояние_абонента', 
    'point_lat', 'point_lon', 'address', 'cluster_name', 'source_file',
    'distance_meters_geo', 'distance_meters_adsl',
    'location_flag_geo', 'location_flag_adsl',
    'usluga_tech', 'rn'
]
cols = [c for c in cols if c in df_comparison.columns]

df_final = df_comparison[cols]

try:
    df_final.to_excel(OUTPUT_XLSX, index=False)
    logging.info(f"Файл сохранен: {OUTPUT_XLSX}")
except Exception as e:
    logging.error(f"Ошибка при сохранении: {e}")


2026-04-28 13:35:07,787 - INFO - Подготовка и сохранение результатов...
2026-04-28 13:35:22,222 - INFO - Файл сохранен: /Users/narikenzhe/Desktop/My_Jupyter/df_comparison_20260428_133437.xlsx


In [8]:
# 5. Визуализация результатов на карте (генерация HTML)
import folium
from folium.plugins import FastMarkerCluster
import numpy as np

# Используем только что созданный OUTPUT_XLSX (или укажите вручную)
try:
    file_path = OUTPUT_XLSX
except NameError:
    file_path = '/Users/narikenzhe/Desktop/My_Jupyter/df_comparison_20260427_165546.xlsx'

df_vis = pd.read_excel(file_path)

# ИСКЛЮЧАЕМ (на всякий случай, если файл старый)
df_vis = df_vis[df_vis['состояние_абонента'] != 'В кластере (df_geo), но нет в ADSL'].copy()

# Объединенный флаг локации (внутри или рядом)
df_vis['location_flag'] = df_vis['location_flag_geo'].fillna(df_vis['location_flag_adsl']).fillna('неизвестно')

# Готовим HTML для попапов (информация при клике на точку)
df_vis['popup'] = (
    "<div style='width: 250px; font-family: sans-serif;'>" +
    "<b>Состояние:</b> " + df_vis['состояние_абонента'].astype(str).fillna('') + "<br>" +
    "<b>Локация к кластеру:</b> <span style='color: #007bff;'>" + df_vis['location_flag'].astype(str).str.upper() + "</span><br>" +
    "<b>Customer Account ID:</b> <span style='color: #d9534f; font-weight: bold;'>" + df_vis['customer_account_id'].fillna('').astype(str) + "</span><br>" +
    "<b>Адрес:</b> " + df_vis['address'].fillna('').astype(str) + "<br>" +
    "<b>Кластер:</b> " + df_vis.get('cluster_name', df_vis['source_file']).fillna('').astype(str) + "<br>" +
    "<b>Услуга:</b> " + df_vis['usluga_tech'].fillna('').astype(str) +
    "</div>"
)

# Центрируем карту
map_center = [df_vis['point_lat'].mean(), df_vis['point_lon'].mean()]
m = folium.Map(location=map_center, zoom_start=12)

# Добавляем границы кластеров (полигоны)
if 'df_clusters' in locals() or 'df_clusters' in globals():
    fg_clusters = folium.FeatureGroup(name="Границы кластеров", show=True)
    folium.GeoJson(
        df_clusters,
        style_function=lambda feature: {
            'fillColor': feature['properties'].get('fill', '#3388ff'),
            'color': feature['properties'].get('stroke', '#3388ff'),
            'weight': 2,
            'fillOpacity': 0.2
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['name', 'source_file'], 
            aliases=['Кластер:', 'Файл:']
        )
    ).add_to(fg_clusters)
    m.add_child(fg_clusters)

# JS-колбэк для FastMarkerCluster, чтобы привязывать popup к каждому маркеру
callback = """
function (row) {
    var icon = L.AwesomeMarkers.icon({
        icon: 'info-sign',
        iconColor: 'white',
        markerColor: 'blue',
        prefix: 'glyphicon',
        extraClasses: 'fa-rotate-0'
    });
    var marker = L.marker(new L.LatLng(row[0], row[1]), {icon: icon});
    marker.bindPopup(row[2]);
    return marker;
};
"""

# Разделяем по состояниям для разной визуализации (отключаемые слои)
states = df_vis['состояние_абонента'].unique()

for state in states:
    state_df = df_vis[df_vis['состояние_абонента'] == state].dropna(subset=['point_lat', 'point_lon'])
    points = state_df[['point_lat', 'point_lon', 'popup']].values.tolist()
    
    fg = folium.FeatureGroup(name=state)
    cluster = FastMarkerCluster(data=points, callback=callback, name=state)
    fg.add_child(cluster)
    m.add_child(fg)

folium.LayerControl().add_to(m)

# Добавляем метрики в правом нижнем углу
metrics_html = """
<div style='position: fixed; bottom: 50px; right: 50px; width: auto; max-width: 450px; height: auto; 
            background-color: white; border: 2px solid #ccc; border-radius: 5px; z-index:9999; 
            font-size:14px; padding: 15px; box-shadow: 2px 2px 5px rgba(0,0,0,0.3); font-family: sans-serif;'>
    <h4 style='margin-top: 0; margin-bottom: 10px;'>Общие метрики</h4>
"""

grouped_counts = df_vis.groupby(['состояние_абонента', 'location_flag']).size().reset_index(name='count')
for state in states:
    state_data = grouped_counts[grouped_counts['состояние_абонента'] == state]
    total_state = state_data['count'].sum()
    if total_state == 0: continue
    
    metrics_html += f"<div style='margin-bottom: 8px;'><b>{state}</b> ({total_state}):<ul style='margin-top: 3px; margin-bottom: 3px;'>"
    for _, row in state_data.iterrows():
        loc = row['location_flag']
        cnt = row['count']
        metrics_html += f"<li style='margin-left: -20px;'>{loc.capitalize()}: {cnt}</li>"
    metrics_html += "</ul></div>"

metrics_html += f"<div style='margin-top: 10px; border-top: 1px solid #eee; padding-top: 5px;'>Всего точек: <b>{len(df_vis)}</b></div>"
metrics_html += "</div>"

m.get_root().html.add_child(folium.Element(metrics_html))

# Сохраняем в HTML
output_html = '/Users/narikenzhe/Desktop/My_Jupyter/map_comparison.html'
m.save(output_html)
print(f"Карта успешно сохранена в {output_html}\nОткройте этот файл в браузере для просмотра.")


FileNotFoundError: [Errno 2] No such file or directory: '/Users/narikenzhe/Desktop/My_Jupyter/df_comparison_20260427_165546.xlsx'